## Luật chơi Othello

- Othello được chơi trên bàn cờ **8×8** với hai loại quân: **Đen** và **Trắng**.
- Ban đầu có **4 quân ở trung tâm**, gồm 2 quân Đen và 2 quân Trắng đặt xen kẽ theo đường chéo.
- **Đen đi trước**, sau đó hai bên lần lượt thực hiện lượt chơi.
- Một nước đi hợp lệ phải đặt quân vào ô trống sao cho **kẹp ít nhất một quân đối phương** theo hàng ngang, dọc hoặc chéo.
- Các quân đối phương bị kẹp sẽ được **lật thành màu của người chơi hiện tại**.
- Nếu người chơi không có nước đi hợp lệ thì phải **bỏ lượt (pass)**.

### Điều kiện kết thúc

Trò chơi kết thúc khi **cả hai người chơi đều không còn nước đi hợp lệ**. Các trường hợp thường gặp:

1. Bàn cờ đã đầy.
2. Một bên không còn quân.
3. Cả hai bên đều không thể đi tiếp dù bàn cờ vẫn còn ô trống.

Người có **nhiều quân hơn** khi trò chơi kết thúc sẽ chiến thắng.

### Khởi tạo bàn cờ 
- Dùng mảng 8x8:
    - `0` - trống
    - `1` - đen
    - `-1` - trắng
- Khởi tạo 4 quân giữa bàn cờ và hàm in ra màn hình

In [56]:
size = 8

empty = 0
black = 1
white = -1

symbols = {empty: ".", black: "●", white: "○"}

def init_board():
    board = [[empty] * size for _ in range(size)]

    mid = size // 2 # chia kích thước bàn cờ thành 4x4
    board[mid-1][mid-1] = white #d4
    board[mid][mid] = white # e5
    board[mid-1][mid] = black #e4
    board[mid][mid-1] = black # d5
    
    return board


def print_board(board):
    print("  "+ " ".join("abcdefgh"[:size]))
    for r in range(size):
        row = " ".join(symbols[board[r][c]] for c in range(size))
        print(f"{r+1} {row}")
        
#hàm đếm số lượng quân cờ trên bàn cờ
def count_pieces(board):
    count_black = sum(row.count(black) for row in board)
    count_white = sum(row.count(white) for row in board)
    
    print(f"Black: {count_black}, White: {count_white}")
    return count_black, count_white

In [57]:
board = init_board()
print_board(board)
print(count_pieces(board))

  a b c d e f g h
1 . . . . . . . .
2 . . . . . . . .
3 . . . . . . . .
4 . . . ○ ● . . .
5 . . . ● ○ . . .
6 . . . . . . . .
7 . . . . . . . .
8 . . . . . . . .
Black: 2, White: 2
(2, 2)


### Kiểm tra nước đi hợp lệ 
- `get_flips(board, row, col, player)`: từ ô định đặt, đi theo 8 hướng, gom các quân đối thủ liền nhau. Nếu cuối dãy gặp quân mình thì lật lại.
- Nước đi hợp lệ là nước đi có ít nhất 1 quân địch bị kẹp.
- `get_valid_moves(board, player)`: trả về các nước đi hợp lệ có thể đi được


In [58]:
#Hướng di chuyển của cờ tại ô hiện tại (row, col) theo 8 hướng
from pandas import col


directions= [(-1, -1), (-1, 0), (-1, 1), 
             (0, -1),            (0, 1), 
             (1, -1),   (1, 0),  (1, 1)]

#hàm này dùng để kiểm tra xem một ô có nằm trong phạm vi bàn cờ hay không
def in_bounds(row, col):
    return 0 <= row < size and 0 <= col < size

def get_flips(board, row, col, player):
    if board[row][col] != empty:
        return []  # nếu ô này đã có cờ, không thể đặt
    
    if player == black:
        opponent = white
    else:
        opponent = black
        
    flips=[]
    for dr, dc in directions:
        r, c = row + dr, col + dc
        temp_flips=[]
        while in_bounds(r, c) and board[r][c] == opponent:
            temp_flips.append((r, c))
            r += dr
            c += dc
        if in_bounds(r, c) and board[r][c] == player:
            flips.extend(temp_flips)
    return flips

def get_valid_moves(board, player):
    moves = []
    for r in range(size):
        for c in range(size):
            if get_flips(board, r, c, player):
                moves.append((r, c))
    return moves

def to_notation(row, col):
    """Đổi (r, c) sang ký hiệu cờ, ví dụ (2, 3) -> 'd3'."""
    return f"{'abcdefgh'[col]}{row + 1}"

def print_valid_moves(board, player):
    moves = get_valid_moves(board, player)
    move_set = set(moves)
    print("  "+ " ".join("abcdefgh"[:size]))
    for row in range(size):
        cell = []
        for col in range(size):
            if(row, col) in move_set:
                cell.append("*")
            else:
                cell.append(symbols[board[row][col]])
        print(f"{row+1} " + " ".join(cell))
        
    if moves:
        print("Moves:", ", ".join(to_notation(r, c) for r, c in moves))
    else:
        print("No valid moves. Next")

In [59]:
print(f"Black move: {get_valid_moves(board, black)}")
print_valid_moves(board, black)


Black move: [(2, 3), (3, 2), (4, 5), (5, 4)]
  a b c d e f g h
1 . . . . . . . .
2 . . . . . . . .
3 . . . * . . . .
4 . . * ○ ● . . .
5 . . . ● ○ * . .
6 . . . . * . . .
7 . . . . . . . .
8 . . . . . . . .
Moves: d3, c4, f5, e6


In [60]:
print(f"White move: {get_valid_moves(board, white)}")
print_valid_moves(board, white)

White move: [(2, 4), (3, 5), (4, 2), (5, 3)]
  a b c d e f g h
1 . . . . . . . .
2 . . . . . . . .
3 . . . . * . . .
4 . . . ○ ● * . .
5 . . * ● ○ . . .
6 . . . * . . . .
7 . . . . . . . .
8 . . . . . . . .
Moves: e3, f4, c5, d6


### Thực hiện nước đi
- hàm `make_move(board, row, col, player)`: đặt quân và lật tất cả các quân trong hàm `get_flips`

In [61]:
def make_move(board, row, col, player):
    flips = get_flips(board, row, col, player)
    if not flips:
        return None  # move không hợp lệ
    
    #tạo bàn cờ mới để lưu các nước đi 
    new_board=[row[:] for row in board]
    new_board[row][col] =player
    for r, c in flips:
        new_board[r][c] = player
    return new_board    

In [62]:
board = make_move(board, 2, 3, black)
print_board(board)
print(count_pieces(board))
print(f"Black move: {get_valid_moves(board, black)}")
print("\n")
print(f"White move: {get_valid_moves(board, white)}")
print_valid_moves(board, white)

  a b c d e f g h
1 . . . . . . . .
2 . . . . . . . .
3 . . . ● . . . .
4 . . . ● ● . . .
5 . . . ● ○ . . .
6 . . . . . . . .
7 . . . . . . . .
8 . . . . . . . .
Black: 4, White: 1
(4, 1)
Black move: [(4, 5), (5, 4), (5, 5)]


White move: [(2, 2), (2, 4), (4, 2)]
  a b c d e f g h
1 . . . . . . . .
2 . . . . . . . .
3 . . * ● * . . .
4 . . . ● ● . . .
5 . . * ● ○ . . .
6 . . . . . . . .
7 . . . . . . . .
8 . . . . . . . .
Moves: c3, e3, c5


In [63]:
board = make_move(board, 2, 2, white)
print_board(board)
print(count_pieces(board))

  a b c d e f g h
1 . . . . . . . .
2 . . . . . . . .
3 . . ○ ● . . . .
4 . . . ○ ● . . .
5 . . . ● ○ . . .
6 . . . . . . . .
7 . . . . . . . .
8 . . . . . . . .
Black: 3, White: 3
(3, 3)


### Vòng lặp trò chơi
- Luân phiên theo lượt (quân đen phải bắt đầu trước). Bên không có nước đi hợp lệ thì bỏ lượt.
- kết thúc là khi cả 2 bên đều không đi được, sau đó đếm số quân, bên nào nhiều hơn, thì bên đó thắng

### Nhập nước đi 
- Nhập nước đi trên terminal trong phạm vị 8x8 nếu nhập nước đi lớn hơn thì báo lỗi.
- Có thể thêm AI để Human đấu kháng với AI

### Giao diện đồ họa 
- Dùng thư viện `pygame` hoặc `tkinter`